In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

#Partie CDC

In [0]:
import sys; sys.path.append("../generators")
from cdc_simulator import write_batch, describe
describe()

In [0]:
#generate first batch

write_batch(1)

In [0]:
%sql

CREATE OR REPLACE TEMPORARY TABLE  tmp_customers AS 
SELECT *
FROM read_files(
    "/Volumes/bank/ops/landing/cdc/customers/batch_01.json",
    format => "json");

DESCRIBE EXTENDED tmp_customers;

In [0]:
%sql

CREATE OR REPLACE TEMPORARY TABLE  tmp_accounts AS 
SELECT *
FROM read_files(
    "/Volumes/bank/ops/landing/cdc/accounts/batch_01.json",
    format => "json");

DESCRIBE EXTENDED tmp_accounts;

In [0]:
%sql
SELECT count(*) FROM tmp_customers;
SELECT count(*) FROM tmp_accounts;

In [0]:
#Launch batch 2

write_batch(2)

In [0]:
%sql
---check status of the silver customer table

SELECT customer_id, count(*) AS n_versions, count_if(__END_AT IS NULL) AS n_current
FROM bank.silver.customers
GROUP BY ALL
HAVING count(*) > 1 OR count_if(__END_AT IS NULL) = 0
ORDER BY 1;



In [0]:
%sql
SELECT
*
FROM  bank.silver.customers_current
WHERE customer_id = 'C001';

In [0]:
# write 3rd batch 
write_batch(3)

In [0]:
%sql
SELECT *
FROM read_files(
    "/Volumes/bank/ops/landing/cdc/customers/batch_03.json",
    format => "json");

In [0]:
%sql
SELECT customer_id, address, __START_AT, __END_AT
FROM bank.silver.customers WHERE customer_id = 'C004' ORDER BY __START_AT;

In [0]:
%sql
SELECT *
FROM read_files(
    "/Volumes/bank/ops/landing/cdc/accounts/batch_03.json",
    format => "json");

#Partie Fraude Detection


In [0]:
from fraud_scenarios import window_start, inject_all_scenarios, push_watermark
ws = window_start(); print(ws)       
inject_all_scenarios(ws)

In [0]:
%sql
SELECT 
*
FROM read_files(
    '/Volumes/bank/ops/landing/transactions/lab3_CARD-VELO_520e72.json',
    format => 'json'
)

In [0]:
import sys
sys.path.append("../streaming")
from fraud_detection import make_upsert, TARGET

In [0]:
%sql
--Après ingestion et lancement de fraud_alerts.py

SELECT * FROM bank.gold.fraud_alerts 
WHERE card_id LIKE 'CARD-%' ORDER BY card_id;

In [0]:
%sql
DESCRIBE HISTORY bank.gold.fraud_alerts

In [0]:

def n_alerts():
    return spark.table(TARGET).where("card_id LIKE 'CARD-%'").count()

print("avant :", n_alerts())
replay = spark.table(TARGET).where("card_id LIKE 'CARD-%'")
make_upsert(TARGET)(replay, 999)
make_upsert(TARGET)(replay, 999)
print("après :", n_alerts())

In [0]:
%sql
DESCRIBE HISTORY bank.gold.fraud_alerts LIMIT 3;

In [0]:
q = run(output_mode="append", checkpoint="/Volumes/bank/ops/checkpoints/fraud_append",
        target="bank.gold.fraud_alerts_append")
q.awaitTermination()
for p in q.recentProgress:
    print(p["batchId"], p["numInputRows"], p.get("eventTime", {}).get("watermark"))

In [0]:
%sql
SELECT * FROM bank.gold.fraud_alerts_append 
WHERE card_id LIKE 'CARD-%' ORDER BY card_id;

In [0]:
from datetime import datetime
ws=datetime.fromisoformat('2026-11-20 13:10:00+00:00')  # le même ws qu'à l'étape 
push_watermark(ws)        #push a event to move ws to ws+20 


In [0]:
q = run(output_mode="append", checkpoint="/Volumes/bank/ops/checkpoints/fraud_append",
        target="bank.gold.fraud_alerts_append")
q.awaitTermination()
for p in q.recentProgress:
    print(p["batchId"], p["numInputRows"], p.get("eventTime", {}).get("watermark"))

In [0]:
%sql
SELECT * FROM bank.gold.fraud_alerts_append 
WHERE card_id LIKE 'CARD-%' ORDER BY card_id;

In [0]:
import importlib, fraud_detection

q = fraud_detection.run(output_mode="append",
                        checkpoint="/Volumes/bank/ops/checkpoints/fraud_sliding",
                        target="bank.gold.fraud_alerts_sliding")
q.awaitTermination()
for p in q.recentProgress:
    print(p["batchId"], p["numInputRows"], p.get("eventTime", {}).get("watermark"))

In [0]:
%sql
SELECT card_id, window_start, window_end, n_tx, total
FROM bank.gold.fraud_alerts_sliding
WHERE card_id LIKE 'CARD-%' ORDER BY card_id, window_start;

In [0]:
%sql
SELECT * FROM bank.gold.fraud_alerts_sliding 
WHERE card_id LIKE 'CARD-%' ORDER BY card_id;